# Lesson 1 — What is Model Context Protocol?\n\nThis notebook models the MCP architectural boundary with plain Python. It is **not** an MCP implementation yet: the goal is to make hosts, clients, servers, tools, resources, prompts, discovery, and host-owned policy explicit before using an SDK.

## The mental model\n\nMCP is a protocol between an **MCP client** in an AI host and an **MCP server**. The server exposes selected capabilities. The host still owns product policy, identity, consent, and execution authority.

In [ ]:
from dataclasses import dataclass\nfrom typing import Callable\n\n@dataclass(frozen=True)\nclass Tool:\n    name: str\n    description: str\n\n@dataclass(frozen=True)\nclass Resource:\n    uri: str\n    description: str\n\n@dataclass(frozen=True)\nclass Prompt:\n    name: str\n    description: str\n\n@dataclass(frozen=True)\nclass McpServer:\n    name: str\n    tools: tuple[Tool, ...]\n    resources: tuple[Resource, ...]\n    prompts: tuple[Prompt, ...]\n

## Discovery is capability metadata\n\nA real client and server exchange protocol messages. This small client models the outcome: the host receives a structured capability catalogue. No model is involved and no permission has been granted yet.

In [ ]:
@dataclass(frozen=True)\nclass CapabilityCatalogue:\n    server_name: str\n    tools: tuple[Tool, ...]\n    resources: tuple[Resource, ...]\n    prompts: tuple[Prompt, ...]\n\nclass McpClient:\n    def __init__(self, name: str) -> None:\n        self.name = name\n\n    def discover(self, server: McpServer) -> CapabilityCatalogue:\n        return CapabilityCatalogue(\n            server_name=server.name,\n            tools=server.tools,\n            resources=server.resources,\n            prompts=server.prompts,\n        )\n

In [ ]:
server = McpServer(\n    name='repository-context',\n    tools=(Tool('list_open_pull_requests', 'List open pull requests in an approved repository.'),),\n    resources=(Resource('repo://engineering-guide', 'Engineering guide for this repository.'),),\n    prompts=(Prompt('summarise_pull_request', 'Summarise a pull request for a reviewer.'),),\n)\n\nclient = McpClient(name='course-host')\ncatalogue = client.discover(server)\n\nprint(f'Connected {client.name} → {catalogue.server_name}')\nprint('Tools:', [tool.name for tool in catalogue.tools])\nprint('Resources:', [resource.uri for resource in catalogue.resources])\nprint('Prompts:', [prompt.name for prompt in catalogue.prompts])

## Discovery is not authorisation\n\nThe server's tool list is a proposal of what exists. The host must decide which tools are eligible for the current user and run. That check is deterministic application policy, not model judgment.

In [ ]:
def host_allows_tool(*, user_role: str, tool_name: str) -> bool:\n    allowed_by_role = {\n        'reader': {'list_open_pull_requests'},\n        'maintainer': {'list_open_pull_requests', 'merge_pull_request'},\n    }\n    return tool_name in allowed_by_role.get(user_role, set())\n\nfor requested_tool in ('list_open_pull_requests', 'merge_pull_request'):\n    print(\n        f'reader requesting {requested_tool}:',\n        'allowed' if host_allows_tool(user_role='reader', tool_name=requested_tool) else 'denied',\n    )

## MCP versus function calling\n\nMCP discovers and communicates with external capability providers. Function calling is how an LLM may select one of the host's permitted tools. The two can work together, but they solve different problems.\n\n**Lesson 1 takeaway:** MCP standardises connectivity; the host retains security and execution control.